In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df_samples = pd.read_csv("/home/earkfeld/Projects/MitoSpace4D/runs/embeddings/image_paths.csv", dtype=str, header=None, delimiter=" ")
df_regions = pd.read_csv("/run/user/1002/gvfs/smb-share:server=aquila0.jslab.ucsd.edu,share=ssd_processing/Others/MitoSpace4D/2025_summer/cell_to_region.csv")

In [3]:
# Set the header for the first column
df_samples.columns = ["fpath"]
df_samples['condition'] = df_samples['fpath'].apply(lambda x: x.split("/")[-2])
df_samples['filename'] = df_samples['fpath'].apply(lambda x: x.split("/")[-1].split(".")[0])
df_samples['cell_id'] = df_samples['filename'].apply(lambda x: int(x.split("-")[0]))
df_samples['cell_tid'] = df_samples['filename'].apply(lambda x: int(x.split("-")[1]))

# Set up an empty column called "condition_tid"
df_samples['region_id'] = -1

In [4]:
for condition in df_regions["Data Path"].unique():
    df_regions_condition = df_regions[df_regions["Data Path"] == condition]
    df_samples_condition = df_samples[df_samples["condition"] == condition]

    for i, row in df_regions_condition.iterrows():
        cell_id_start = row["Cell ID Start"]
        try:
            cell_id_end = df_regions_condition.loc[i + 1, "Cell ID Start"]
        except:
            # Set infinite integer value
            cell_id_end = np.inf

        current_region_id = row["Region ID"]
        for i, sample_row in df_samples_condition.iterrows():
            current_cell_id = sample_row['cell_id']

            if current_cell_id >= cell_id_start and current_cell_id < cell_id_end:
                df_samples_condition.at[i, "region_id"] = current_region_id

    df_samples.update(df_samples_condition)

In [5]:
time_id_fn = lambda region_id, cell_tid: 3*region_id + cell_tid
df_samples["time_id"] = df_samples.apply(lambda x: time_id_fn(x["region_id"], x["cell_tid"]), axis=1)

In [6]:
# Set up column for next instance index 
df_samples["next_iindex"] = int(-1)

for i, row in df_samples.iterrows():
    condition = row["condition"]
    region_id = row["region_id"]
    cell_id = row["cell_id"]
    cell_tid = row["cell_tid"]

    if cell_tid == 2:
        df_samples.at[i, "next_iindex"] = int(-1)

    # find the next instance (same condition, region_id, and cell_id with cell_tid+1)
    df_filtered = df_samples[(df_samples["condition"] == condition) &
                              (df_samples["region_id"] == region_id) &
                              (df_samples["cell_id"] == cell_id) &
                              (df_samples["cell_tid"] == cell_tid + 1)]
    if not df_filtered.empty:
        df_samples.at[i, "next_iindex"] = int(df_filtered.index[0])

In [ ]:
# df_samples.sort_values(by=["condition", "filename", "time_id"], inplace=False)

In [7]:
# Get the indices for each row
curr_iindex_arr = np.vstack(df_samples.index.values)
next_iindex_arr = np.vstack(df_samples["next_iindex"].values)

iindex_arr = np.concatenate((curr_iindex_arr, next_iindex_arr), axis=1)

In [8]:
# Save the array
np.save("instance_vector_indices.npy", iindex_arr)

In [ ]:
# Set up a plasma color map as a function of time_id (rgb, 0-255)
df_samples['color'] = df_samples['time_id'].apply(lambda x: (np.array(plt.cm.plasma(x / df_samples['time_id'].max())[:3]) * 255).astype(int))

In [ ]:
colors_arr = np.vstack(df_samples['color'].values)
np.save('temporal_colormap.npy', colors_arr)